# 04 — Clean CPS Data

This notebook cleans the IPUMS CPS March ASEC supplement for the heterogeneity analysis.
The goal is to produce a state × year panel of **employment rates** broken down by
age group and gender,  to test whether minimum wage effects are concentrated among
specific demographic groups (teens, young adults, women).

**Input:** `data/raw/cps/cps_00001.csv.gz`

**Output:**
- `data/intermediate/cps_clean.parquet` — individual-level 
- `data/intermediate/cps_panel.parquet` — state × year × group employment rates

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
RAW_CPS = ROOT / "data" / "raw" / "cps"
INTERMEDIATE = ROOT / "data" / "intermediate"
INTERMEDIATE.mkdir(parents=True, exist_ok=True)

CPS_FILE = RAW_CPS / "cps_00001.csv.gz"
OUTPUT_CLEAN = INTERMEDIATE / "cps_clean.parquet"
OUTPUT_PANEL = INTERMEDIATE / "cps_panel.parquet"

FIRST_YEAR = 2004
LAST_YEAR = 2024

## 1. Load and inspect the raw file

In [2]:
raw = pd.read_csv(CPS_FILE, dtype={"STATEFIP": str})

print("Shape  :", raw.shape)
print("Dtypes :")
print(raw.dtypes)
print("\nHead:")
raw.head()

Shape  : (3967030, 19)
Dtypes :
YEAR           int64
SERIAL         int64
MONTH          int64
CPSID          int64
ASECFLAG       int64
HFLAG        float64
ASECWTH      float64
STATEFIP         str
PERNUM         int64
CPSIDP         int64
CPSIDV         int64
ASECWT       float64
AGE            int64
SEX            int64
EMPSTAT        int64
LABFORCE       int64
UHRSWORKT      int64
WKSWORK2       int64
INCWAGE        int64
dtype: object

Head:


,YEAR,SERIAL,MONTH,CPSID,ASECFLAG,HFLAG,ASECWTH,STATEFIP,PERNUM,CPSIDP,CPSIDV,ASECWT,AGE,SEX,EMPSTAT,LABFORCE,UHRSWORKT,WKSWORK2,INCWAGE
0,2004,5,3,20040106130600,1,NaN,538.13,23,1,20040106130601,200401061306011,538.13,59,2,10,2,40,6,60000
1,2004,6,3,20040202839800,1,NaN,613.24,23,1,20040202839801,200402028398011,613.24,49,1,10,2,20,6,32000
2,2004,6,3,20040202839800,1,NaN,613.24,23,2,20040202839802,200402028398021,679.17,19,1,34,1,999,0,0
3,2004,8,3,20040106130700,1,NaN,652.28,23,1,20040106130701,200401061307011,652.28,42,2,10,2,40,6,30000
4,2004,8,3,20040106130700,1,NaN,652.28,23,2,20040106130702,200401061307021,652.28,42,1,10,2,40,6,0


In [3]:
print("Year range    :", raw["YEAR"].min(), "to", raw["YEAR"].max())
print("Unique states :", raw["STATEFIP"].nunique())
print("\nMissing values:")
print(raw.isnull().sum())
print("\nASECFLAG value counts:")
print(raw["ASECFLAG"].value_counts())
print("\nLABFORCE value counts:")
print(raw["LABFORCE"].value_counts())
print("\nEMPSTAT value counts:")
print(raw["EMPSTAT"].value_counts().sort_index())

Year range    : 2004 to 2024
Unique states : 51

Missing values:
YEAR               0
SERIAL             0
MONTH              0
CPSID              0
ASECFLAG           0
HFLAG        3767474
ASECWTH            0
STATEFIP           0
PERNUM             0
CPSIDP             0
CPSIDV             0
ASECWT             0
AGE                0
SEX                0
EMPSTAT            0
LABFORCE           0
UHRSWORKT          0
WKSWORK2           0
INCWAGE            0
dtype: int64

ASECFLAG value counts:
ASECFLAG
1    3967030
Name: count, dtype: int64

LABFORCE value counts:
LABFORCE
2    1941997
1    1102007
0     923026
Name: count, dtype: int64

EMPSTAT value counts:
EMPSTAT
0      909484
1       13542
10    1766731
12      61175
21     104045
22      10046
32     148451
34     503533
36     450023
Name: count, dtype: int64


## 2. Filter to March ASEC, working-age adults in the labor force

In [4]:
n_raw = len(raw)

# Keep only March ASEC observations
df = raw[raw["ASECFLAG"] == 1].copy()
print(f"After ASEC filter         : {len(df):,} (dropped {n_raw - len(df):,})")

# Analysis years only
df = df[df["YEAR"].between(FIRST_YEAR, LAST_YEAR)]
print(f"After year filter         : {len(df):,}")

# Working-age adults (16–64)
df = df[df["AGE"].between(16, 64)]
print(f"After age filter (16–64)  : {len(df):,}")

# In the labor force
df = df[df["LABFORCE"] == 2]
print(f"After labor force filter  : {len(df):,}")

After ASEC filter         : 3,967,030 (dropped 0)
After year filter         : 3,967,030
After age filter (16–64)  : 2,502,460
After labor force filter  : 1,844,510


## 3. Create analysis variables

In [5]:
# Employed = at work or has job but not at work
df["employed"] = df["EMPSTAT"].isin([10, 12]).astype(int)

# Gender
df["female"] = (df["SEX"] == 2).astype(int)

# Age groups — standard low-wage worker categories
df["age_group"] = pd.cut(
    df["AGE"],
    bins=[15, 19, 24, 34, 44, 54, 64],
    labels=["16-19", "20-24", "25-34", "35-44", "45-54", "55-64"],
)
df["age_group"] = df["age_group"].astype(str)

# Zero-pad state FIPS to 2 digits
df["state_fips"] = df["STATEFIP"].str.zfill(2)

# Year and weight
df["year"] = df["YEAR"].astype(int)
df["weight"] = df["ASECWT"].astype(float)

print("Age group distribution:")
print(df["age_group"].value_counts().sort_index())
print(
    f"\nOverall employment rate (weighted): {np.average(df['employed'], weights=df['weight']):.3f}"
)

Age group distribution:
age_group
16-19     86641
20-24    160291
25-34    408754
35-44    472125
45-54    440131
55-64    276568
Name: count, dtype: int64

Overall employment rate (weighted): 0.939


## 4. Handle NIU / missing codes

In [6]:
# UHRSWORKT: 999 = NIU
df["UHRSWORKT"] = df["UHRSWORKT"].replace(999, np.nan)

# INCWAGE: 999999 = NIU, 999998 = missing
df["INCWAGE"] = df["INCWAGE"].replace({999999: np.nan, 999998: np.nan})

print("Missing values after NIU handling:")
print(
    df[["UHRSWORKT", "INCWAGE", "employed", "female", "age_group", "weight"]]
    .isnull()
    .sum()
)

Missing values after NIU handling:
UHRSWORKT    109555
INCWAGE           0
employed          0
female            0
age_group         0
weight            0
dtype: int64


## 5. Select final columns and save individual-level file

In [7]:
KEEP_COLS = [
    "state_fips",
    "year",
    "AGE",
    "age_group",
    "female",
    "employed",
    "INCWAGE",
    "UHRSWORKT",
    "weight",
]

cps_clean = df[KEEP_COLS].copy().reset_index(drop=True)

print("Individual-level file shape:", cps_clean.shape)
cps_clean.to_parquet(OUTPUT_CLEAN, index=False)

Individual-level file shape: (1844510, 9)


## 6. Collapse to state × year × age_group × gender employment rates

We compute **weighted** employment rates for each cell. These will be merged with the
minimum wage panel in the heterogeneity notebook.

In [8]:
def weighted_mean(group):
    w = group["weight"]
    return pd.Series(
        {
            "emp_rate": np.average(group["employed"], weights=w),
            "n_individuals": len(group),
            "total_weight": w.sum(),
        }
    )


# Full breakdown: state × year × age_group × female
cps_panel = (
    cps_clean.groupby(["state_fips", "year", "age_group", "female"])
    .apply(weighted_mean)
    .reset_index()
)

# Also compute state × year × age_group (both genders combined)
cps_by_age = (
    cps_clean.groupby(["state_fips", "year", "age_group"])
    .apply(weighted_mean)
    .reset_index()
)
cps_by_age["female"] = -1  # sentinel: both genders

# Stack
cps_panel = pd.concat([cps_panel, cps_by_age], ignore_index=True)
cps_panel["n_individuals"] = cps_panel["n_individuals"].astype(int)

print("Collapsed panel shape:", cps_panel.shape)
print("\nSample:")
cps_panel.head(12)

Collapsed panel shape: (19278, 7)

Sample:


,state_fips,year,age_group,female,emp_rate,n_individuals,total_weight
0,01,2004,16-19,0,0.824236,28,45902.84
1,01,2004,16-19,1,0.923777,39,51035.57
2,01,2004,20-24,0,0.894170,59,114027.51
3,01,2004,20-24,1,0.807976,66,119515.71
4,01,2004,25-34,0,0.950883,155,247350.31
5,01,2004,25-34,1,0.953937,161,226851.35
6,01,2004,35-44,0,0.950307,173,251789.97
7,01,2004,35-44,1,0.940781,185,246960.02
8,01,2004,45-54,0,0.966643,184,279195.91
9,01,2004,45-54,1,0.981016,163,253690.01


In [9]:
# Check national employment rate by age group
national = (
    cps_clean.groupby("age_group")
    .apply(lambda g: np.average(g["employed"], weights=g["weight"]))
    .reset_index()
    .rename(columns={0: "emp_rate"})
)
print(f"National weighted employment rate by age group ({FIRST_YEAR}–{LAST_YEAR}):")
print(national.to_string(index=False))

# By gender
by_gender = (
    cps_clean.groupby("female")
    .apply(lambda g: np.average(g["employed"], weights=g["weight"]))
    .reset_index()
    .rename(columns={0: "emp_rate", "female": "is_female"})
)
print("\nNational weighted employment rate by gender:")
print(by_gender.to_string(index=False))

National weighted employment rate by age group (2004–2024):
age_group  emp_rate
    16-19  0.830349
    20-24  0.897551
    25-34  0.936449
    35-44  0.951067
    45-54  0.954069
    55-64  0.956982

National weighted employment rate by gender:
 is_female  emp_rate
         0  0.933809
         1  0.945020


## 7. Save collapsed panel

In [10]:
cps_panel.to_parquet(OUTPUT_PANEL, index=False)